In [5]:
from bisect import bisect_left, bisect_right
from math import inf

def optimal_transmitters(houses, candidates, radius=500):
    houses.sort()
    N = len(houses)
    intervals_set = set()
    
    if N == 0:
        return 0
    
    if not candidates:
        return None
    
    for p in candidates:
        left = bisect_left(houses, p - radius)
        right = bisect_right(houses, p + radius) - 1
        
        if left <= right:
            intervals_set.add((left, right))
        
    intervals = list(intervals_set)
    intervals.sort(key = lambda x: (x[0], x[1]))
    M = len(intervals)
    
    def overlap(i, j):
        Li, Ri = intervals[i]
        Lj, Rj = intervals[j]
        return max(0, min(Ri, Rj) - max(Li, Lj) + 1)
    
    dp = [inf]*M

    for i in range(M):
        Li, Ri = intervals[i]
        if Li == 0:
            dp[i] = 0
        
        j = i-1
        while j >= 0 and intervals[j][1] >= Li-1:
            Lj, Rj = intervals[j]
            if dp[j] != inf:
                dp[i] = min(dp[i], dp[j] + overlap(j, i))
            j -= 1
    
    ans = inf
    k = M-1
    while k >= 0 and intervals[k][1] == N-1:
        ans = min(ans, dp[k])
        k -= 1
    
    if ans == inf: 
        return None
    
    return ans




In [2]:

if __name__ == "__main__":
    test_cases = [
        # Houses, candidate locations, expected overlap count
        ([], [], 0),
        ([0, 2000], [], None),
        ([0, 400, 800], [0, 400, 800], 0),
        ([0, 500, 1000], [0, 1000], 1),
        ([100, 200], [150], 0),
        ([0, 1000], [0, 1000], 0),
        ([200, 400, 600, 800, 1200, 1400, 1500, 1800, 1900], [500, 750, 1000, 1500], 0),
        ([200, 400, 600, 800, 1200, 1400, 1500, 1800, 1900, 2100, 2250], [500, 750, 1000, 1500, 1750], 3),
        ([200, 400, 600, 800, 1200, 1400, 1500, 1800, 1900, 2100, 2300], [500, 750, 1000, 1500, 1750], None),
        ([200, 400, 600, 800, 1200, 1400, 1500, 1800, 1900, 2100, 2300, 2400, 2600, 3000, 3600], [500, 750, 1000, 1500, 1750, 3000, 3100], None)
    ]

    for i, (houses, candidates, expected) in enumerate(test_cases, start=1):
        result = optimal_transmitters(houses[:], candidates[:])

        print(f"Test {i}:")
        print("  Result:", result)
        print("  Expected:", expected)

        if result == expected:
            print("  Passed\n")
        else:
            print("  Failed\n")


Test 1:
  Result: 0
  Expected: 0
  Passed

Test 2:
  Result: None
  Expected: None
  Passed

Test 3:
  Result: 0
  Expected: 0
  Passed

Test 4:
  Result: 1
  Expected: 1
  Passed

Test 5:
  Result: 0
  Expected: 0
  Passed

Test 6:
  Result: 0
  Expected: 0
  Passed

Test 7:
  Result: 0
  Expected: 0
  Passed

Test 8:
  Result: 3
  Expected: 3
  Passed

Test 9:
  Result: None
  Expected: None
  Passed

Test 10:
  Result: None
  Expected: None
  Passed



In [ ]:

from bisect import bisect_left, bisect_right
from math import inf

def optimal_transmitters_parents(houses, candidates, radius=500):
    houses.sort()
    candidates.sort()
    N = len(houses)
    
    if N == 0:
        return 0, []
    
    if not candidates:
        return None, []
    
    intervals_set = set()
    interval_to_transmitter = {}
    
    for p in candidates:
        left = bisect_left(houses, p - radius)
        right = bisect_right(houses, p + radius) - 1
        
        if left <= right:
            interval = (left, right)
            intervals_set.add(interval)
            interval_to_transmitter[interval] = p
            
    intervals = list(intervals_set)
    intervals.sort(key = lambda x: (x[0], x[1]))
    M = len(intervals)
    
    def overlap(i, j):
        Li, Ri = intervals[i]
        Lj, Rj = intervals[j]
        return max(0, min(Ri, Rj) - max(Li, Lj) + 1)
    
    dp = [inf]*M
    parent = [None]*M
    
    for i in range(M):
        Li, Ri = intervals[i]
        if Li == 0:
            dp[i] = 0
            
        j = i-1
        while j >= 0 and intervals[j][1] >= Li-1:
            if dp[j] != inf:
                cost = dp[j] + overlap(j, i)
                
                if cost < dp[i]:
                    dp[i] = cost
                    parent[i] = j
            j -= 1
    
    ans = inf
    end = None
    
    k = M-1
    while k >= 0 and intervals[k][1] == N-1:
        if dp[k] < ans:
            ans = dp[k]
            end = k
        k -= 1
        
    if ans == inf:
        return None, []
    
    selected_transmitters = []
    while end is not None:
        selected_transmitters.append(interval_to_transmitter[intervals[end]])
        end = parent[end]
        
    return ans, selected_transmitters[::-1]


In [4]:
houses = [200, 400, 600, 800, 1200, 1400, 1500, 1800, 1900, 2100, 2250]
candidates = [500, 750, 1000, 1500, 1750]

print(*optimal_transmitters_parents(houses, candidates))

3 [500, 750, 1750]


In [6]:
from bisect import bisect_left, bisect_right
from collections import deque
from math import inf

def optimal_transmitters_fast(houses, candidates, radius=500):
    houses = sorted(houses); candidates = sorted(candidates)
    N = len(houses)
    if N == 0: return 0
    if not candidates: return None
    iv = set()
    for p in candidates:
        l = bisect_left(houses, p - radius)
        r = bisect_right(houses, p + radius) - 1
        if l <= r: iv.add((l, r))
    iv = sorted(iv)
    M = len(iv)
    L = [a for a, _ in iv]; R = [b for _, b in iv]
    dp = [inf] * M
    dq = deque()                     # (dp[j] + R[j], j), values increasing
    lo = 0
    for i in range(M):
        while lo < i and R[lo] < L[i] - 1:   # gap: j can never be valid again
            lo += 1
        while dq and dq[0][1] < lo:
            dq.popleft()
        if L[i] == 0:
            dp[i] = 0
        if dq:
            dp[i] = min(dp[i], dq[0][0] - L[i] + 1)
        if dp[i] < inf:
            v = dp[i] + R[i]
            while dq and dq[-1][0] >= v:
                dq.pop()
            dq.append((v, i))
    ans = min((dp[i] for i in range(M) if R[i] == N - 1), default=inf)
    return None if ans == inf else ans

In [7]:

if __name__ == "__main__":
    test_cases = [
        # Houses, candidate locations, expected overlap count
        ([], [], 0),
        ([0, 2000], [], None),
        ([0, 400, 800], [0, 400, 800], 0),
        ([0, 500, 1000], [0, 1000], 1),
        ([100, 200], [150], 0),
        ([0, 1000], [0, 1000], 0),
        ([200, 400, 600, 800, 1200, 1400, 1500, 1800, 1900], [500, 750, 1000, 1500], 0),
        ([200, 400, 600, 800, 1200, 1400, 1500, 1800, 1900, 2100, 2250], [500, 750, 1000, 1500, 1750], 3),
        ([200, 400, 600, 800, 1200, 1400, 1500, 1800, 1900, 2100, 2300], [500, 750, 1000, 1500, 1750], None),
        ([200, 400, 600, 800, 1200, 1400, 1500, 1800, 1900, 2100, 2300, 2400, 2600, 3000, 3600], [500, 750, 1000, 1500, 1750, 3000, 3100], None)
    ]

    for i, (houses, candidates, expected) in enumerate(test_cases, start=1):
        result = optimal_transmitters(houses[:], candidates[:])

        print(f"Test {i}:")
        print("  Result:", result)
        print("  Expected:", expected)

        if result == expected:
            print("  Passed\n")
        else:
            print("  Failed\n")


Test 1:
  Result: 0
  Expected: 0
  Passed

Test 2:
  Result: None
  Expected: None
  Passed

Test 3:
  Result: 0
  Expected: 0
  Passed

Test 4:
  Result: 1
  Expected: 1
  Passed

Test 5:
  Result: 0
  Expected: 0
  Passed

Test 6:
  Result: 0
  Expected: 0
  Passed

Test 7:
  Result: 0
  Expected: 0
  Passed

Test 8:
  Result: 3
  Expected: 3
  Passed

Test 9:
  Result: None
  Expected: None
  Passed

Test 10:
  Result: None
  Expected: None
  Passed

